# 📊 Phase 4 — Analysis, Gain Evaluation & Publication Visualizations

**Project**: Low-Resource Language (Ukrainian) Sentiment & Emotion Analysis  
**Dataset**: [KSE-RESEARCH-Group/UAReviews](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (11,580 Ukrainian reviews)  
**Embeddings**: Pre-encoded 1024-d dense vectors from [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)  
**Prerequisite**: Run `phase_4.ipynb` to generate `cache/phase4/phase4_sweep_summary.csv` (or use the pre-generated benchmark dataset).  

### Key Research Questions & Visual Deliverables:
1. **Baseline Gain Evaluation**: What are the empirical gains ($\Delta$ Macro F1 and % gain) of each oversampling strategy compared to the raw un-augmented baseline on both the in-distribution `test` set and the out-of-distribution `challenge` set?
2. **Robustness Trade-Off (Clarified Figure 3)**: Clarifying the In-Distribution vs Out-of-Distribution robustness landscape without tangled zig-zag lines, using quadrant crosshairs, baseline anchor, and $\rho$-scaled markers.
3. **Gain Curves vs Imbalance Ratio ($\rho$)**: Tracking gains across $\rho \in \{0.10, 0.25, 0.50, 0.75, 1.00\}$ for `LinearSVC` and `TorchMLP`.
4. **Tail-Class Recovery**: Quantifying F1 recovery for extreme minority emotions (`Fear`: 0.5%, `Surprise`: 0.5%, `Disgust`: 2.1%, `Sadness`: 7.3%).
5. **Domain Generalization Trade-Off**: Direct $\Delta$-$\Delta$ gain scatter mapping where $(0, 0)$ is the baseline.


In [ ]:
# 1. Setup & Publication-Grade Matplotlib Configuration
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Publication styling adhering to standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'legend.fontsize': 8.5,
    'figure.titlesize': 13,
    'svg.fonttype': 'none'
})

STRAT_COLORS = {
    'duplication': '#E66101',    # Vivid amber
    'smote': '#2B83BA',          # Steel cerulean blue
    'smote_renorm': '#008837',   # Forest/emerald green
    'embsmote': '#D7191C'        # Deep crimson/ruby
}

STRAT_LABELS = {
    'duplication': 'Duplication',
    'smote': 'Classic SMOTE',
    'smote_renorm': 'SMOTE + Re-norm',
    'embsmote': 'EmbSMOTE (Inoshita 26)'
}

STRAT_MARKERS = {
    'duplication': 's',
    'smote': 'o',
    'smote_renorm': '^',
    'embsmote': 'D'
}

print("Matplotlib publication style configured.")


In [ ]:
# 2. Robust Storage Setup & Load Sweep Summary
possible_dirs = [
    '/content/drive/MyDrive/lrl_sentiment_short/cache/phase4',
    './phase4',
    '../phase4',
    './lrl_cache/phase4',
    './lrl-sentiment-short/phase4'
]

PHASE4_DIR = None
for d in possible_dirs:
    if os.path.exists(os.path.join(d, 'phase4_sweep_summary.csv')):
        PHASE4_DIR = os.path.abspath(d)
        break

if PHASE4_DIR is None:
    # Fallback to local creation
    PHASE4_DIR = os.path.abspath('./phase4')
    os.makedirs(PHASE4_DIR, exist_ok=True)

summary_csv = os.path.join(PHASE4_DIR, 'phase4_sweep_summary.csv')
print(f"Resolved Phase 4 Directory: {PHASE4_DIR}")
print(f"Loading: {summary_csv}")

df = pd.read_csv(summary_csv)
print(f"Loaded {len(df)} sweep conditions successfully!")
display(df.head(5))


In [ ]:
# 3. Compute Absolute and Relative Gains Relative to Baseline
# Extract baselines (Strategy == 'none')
baselines = df[df['Strategy'] == 'none'].set_index('Classifier')
print("=== RAW BASELINES (No Augmentation, N=8,106) ===")
print(baselines[['Train Size', 'Test Macro F1', 'Test Acc', 'Chal Macro F1', 'Fear F1', 'Surprise F1']])

sweep_df = df[df['Strategy'] != 'none'].copy()

# Compute gains over baseline
sweep_df['Test_Base'] = sweep_df['Classifier'].map(baselines['Test Macro F1'])
sweep_df['Chal_Base'] = sweep_df['Classifier'].map(baselines['Chal Macro F1'])
sweep_df['Test_Gain'] = sweep_df['Test Macro F1'] - sweep_df['Test_Base']
sweep_df['Chal_Gain'] = sweep_df['Chal Macro F1'] - sweep_df['Chal_Base']
sweep_df['Test_Gain_Pct'] = (sweep_df['Test_Gain'] / sweep_df['Test_Base']) * 100
sweep_df['Chal_Gain_Pct'] = (sweep_df['Chal_Gain'] / sweep_df['Chal_Base']) * 100

# Per-class gains for extreme tail classes
for emo in ['Fear', 'Surprise', 'Disgust', 'Sadness']:
    sweep_df[f'{emo}_Base'] = sweep_df['Classifier'].map(baselines[f'{emo} F1'])
    sweep_df[f'{emo}_Gain'] = sweep_df[f'{emo} F1'] - sweep_df[f'{emo}_Base']

print("\nComputed baseline gains. Displaying sample:")
display(sweep_df[['Strategy', 'Rho', 'Classifier', 'Test Macro F1', 'Test_Gain', 'Chal Macro F1', 'Chal_Gain']].head(8))


## 4. Figure 1: Test Set Macro F1 vs Rho Curves

Plots in-distribution Test Macro F1 across oversampling ratios $\rho \in [0.10, 1.00]$. The horizontal dashed line denotes the raw un-augmented baseline.


In [ ]:
# 4. Generate Figure 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

for idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[idx]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf_name]
    base_val = baselines.loc[clf_name, 'Test Macro F1']
    
    ax.axhline(base_val, color='#444444', linestyle='--', linewidth=1.5, zorder=2,
               label=f'Baseline (No Aug, F1={base_val:.4f})')
    
    for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
        s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
        ax.plot(
            s_data['Rho'] * 100, s_data['Test Macro F1'],
            marker=STRAT_MARKERS[strat], color=STRAT_COLORS[strat],
            linewidth=2.2, markersize=8, label=STRAT_LABELS[strat], zorder=4
        )
        
    ax.set_title(f"{clf_name} — In-Distribution Test Macro F1 vs Oversampling Ratio (ρ)", fontweight='bold', pad=10)
    ax.set_xlabel("Target Oversampling Ratio ρ (% of Majority Class)", fontweight='bold')
    ax.set_ylabel("Macro F1", fontweight='bold')
    ax.set_xticks([10, 25, 50, 75, 100])
    ax.set_xticklabels(['10%', '25%', '50%', '75%', '100%'])
    ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig1_path = os.path.join(PHASE4_DIR, 'fig1_macro_f1_vs_rho.png')
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig1_path}")


## 4b. Figure 1b: Out-of-Distribution Challenge Set Macro F1 vs Rho Curves

Plots Out-of-Distribution Challenge Macro F1 across oversampling ratios $\rho \in [0.10, 1.00]$. The horizontal dashed line denotes the raw un-augmented baseline for each classifier.


In [ ]:
# 4b. Generate Figure 1b: Out-of-Distribution Challenge Macro F1 vs Rho
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

for idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[idx]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf_name]
    base_val = baselines.loc[clf_name, 'Chal Macro F1']
    
    ax.axhline(base_val, color='#444444', linestyle='--', linewidth=1.5, zorder=2,
               label=f'Baseline (No Aug, F1={base_val:.4f})')
    
    for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
        s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
        ax.plot(
            s_data['Rho'] * 100, s_data['Chal Macro F1'],
            marker=STRAT_MARKERS[strat], color=STRAT_COLORS[strat],
            linewidth=2.2, markersize=8, label=STRAT_LABELS[strat], zorder=4
        )
        
    ax.set_title(f"{clf_name} — Out-of-Distribution Challenge Macro F1 vs Oversampling Ratio (ρ)", fontweight='bold', pad=10)
    ax.set_xlabel("Target Oversampling Ratio ρ (% of Majority Class)", fontweight='bold')
    ax.set_ylabel("Challenge Macro F1", fontweight='bold')
    ax.set_xticks([10, 25, 50, 75, 100])
    ax.set_xticklabels(['10%', '25%', '50%', '75%', '100%'])
    ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig1_chal_path = os.path.join(PHASE4_DIR, 'fig1_chal_macro_f1_vs_rho.png')
plt.savefig(fig1_chal_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig1_chal_path}")


## 5a. Figure 2a: In-Distribution (Test Set) Tail-Class Recovery Curves

Visualizes class-specific recovery on the in-distribution Test set across the four tail emotions: `Fear` (0.5%), `Surprise` (0.5%), `Disgust` (2.1%), and `Sadness` (7.3%).
On the raw un-augmented baseline, models achieve $0.000$ F1 for `Fear` and `Surprise`.


In [ ]:
# 5a. Generate Figure 2a: Test Set Tail Recovery
tail_emotions = ['Fear', 'Surprise', 'Disgust', 'Sadness']
fig, axes = plt.subplots(len(tail_emotions), 2, figsize=(14, 13), sharex=True, dpi=300)

for row_idx, emo in enumerate(tail_emotions):
    col_name = f"{emo} F1"
    for col_idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
        ax = axes[row_idx, col_idx]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

        sub = sweep_df[sweep_df['Classifier'] == clf_name]
        base_f1 = baselines.loc[clf_name, col_name]
        
        ax.axhline(base_f1, color='#555555', linestyle='--', linewidth=1.2, zorder=2,
                   label=f'Baseline ({base_f1:.3f})')
        
        for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
            s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
            ax.plot(
                s_data['Rho'] * 100, s_data[col_name],
                marker=STRAT_MARKERS[strat], color=STRAT_COLORS[strat],
                linewidth=1.8, markersize=6.5, label=STRAT_LABELS[strat], zorder=4
            )
            
        ax.set_title(f"Test Set — {clf_name}: {emo} Recovery (Baseline: {base_f1:.3f})", fontweight='bold', pad=8)
        ax.set_ylabel(f"{emo} Test F1", fontweight='bold')
        if row_idx == len(tail_emotions) - 1:
            ax.set_xlabel("Oversampling Ratio ρ (% of Majority)", fontweight='bold')
            ax.set_xticks([10, 25, 50, 75, 100])
            ax.set_xticklabels(['10%', '25%', '50%', '75%', '100%'])
        if row_idx == 0 and col_idx == 1:
            ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig2a_path = os.path.join(PHASE4_DIR, 'fig2a_test_tail_class_recovery.png')
plt.savefig(fig2a_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2a_path}")


## 5b. Figure 2b: Out-of-Distribution (Challenge Set) Tail-Class Recovery Curves

Visualizes class-specific recovery on the official Out-of-Distribution Challenge set across `Fear`, `Surprise`, `Disgust`, and `Sadness`.  
Assesses whether minority oversampling produces genuine linguistic generalization or memorization of training artifacts.


In [ ]:
# 5b. Generate Figure 2b: Challenge Set Tail Recovery
fig, axes = plt.subplots(len(tail_emotions), 2, figsize=(14, 13), sharex=True, dpi=300)

for row_idx, emo in enumerate(tail_emotions):
    col_name = f"Chal {emo} F1"
    for col_idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
        ax = axes[row_idx, col_idx]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

        sub = sweep_df[sweep_df['Classifier'] == clf_name]
        base_f1 = baselines.loc[clf_name, col_name]
        
        ax.axhline(base_f1, color='#555555', linestyle='--', linewidth=1.2, zorder=2,
                   label=f'Baseline ({base_f1:.3f})')
        
        for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
            s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
            ax.plot(
                s_data['Rho'] * 100, s_data[col_name],
                marker=STRAT_MARKERS[strat], color=STRAT_COLORS[strat],
                linewidth=1.8, markersize=6.5, label=STRAT_LABELS[strat], zorder=4
            )
            
        ax.set_title(f"Challenge Set — {clf_name}: {emo} Recovery (Baseline: {base_f1:.3f})", fontweight='bold', pad=8)
        ax.set_ylabel(f"{emo} Challenge F1", fontweight='bold')
        if row_idx == len(tail_emotions) - 1:
            ax.set_xlabel("Oversampling Ratio ρ (% of Majority)", fontweight='bold')
            ax.set_xticks([10, 25, 50, 75, 100])
            ax.set_xticklabels(['10%', '25%', '50%', '75%', '100%'])
        if row_idx == 0 and col_idx == 1:
            ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig2b_path = os.path.join(PHASE4_DIR, 'fig2b_chal_tail_class_recovery.png')
plt.savefig(fig2b_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2b_path}")


## 5c. Figure 2c: Class Recovery Heatmap (Tail-Class F1 Recovery Matrix)

Visualizes class recovery as a consolidated matrix comparing all experimental conditions against the un-augmented baseline.  
Shows how extreme minority classes (`Fear`, `Surprise`) transition from $0.000$ to viable decision boundaries, while tracking stability on `Disgust` and `Sadness`.


In [ ]:
# 5c. Generate Figure 2c: Class Recovery Heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 8.5), dpi=300)

tail_emotions = ['Fear', 'Surprise', 'Disgust', 'Sadness']
strat_order = ['duplication', 'smote', 'smote_renorm', 'embsmote']

for c_idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[c_idx]
    sub = sweep_df[sweep_df['Classifier'] == clf_name].copy()
    base_row = df[(df['Classifier'] == clf_name) & (df['Strategy'] == 'none')].iloc[0]
    
    rows_data = []
    r_dict = {'Condition': 'Baseline (No Aug)'}
    for e in tail_emotions:
        r_dict[f"{e} F1"] = base_row[f"{e} F1"]
    r_dict['Macro F1'] = base_row['Test Macro F1']
    rows_data.append(r_dict)
    
    for s in strat_order:
        s_sub = sub[sub['Strategy'] == s].sort_values('Rho')
        for _, r in s_sub.iterrows():
            c_dict = {'Condition': f"{STRAT_LABELS[s]} (ρ={r['Rho']:.2f})"}
            for e in tail_emotions:
                c_dict[f"{e} F1"] = r[f"{e} F1"]
            c_dict['Macro F1'] = r['Test Macro F1']
            rows_data.append(c_dict)
            
    m_df = pd.DataFrame(rows_data).set_index('Condition')
    
    sns.heatmap(
        m_df, ax=ax, cmap='YlGnBu', annot=True, fmt='.3f',
        cbar=c_idx == 1, cbar_kws={'label': 'F1 Score' if c_idx == 1 else ''},
        linewidths=0.5, linecolor='#EEEEEE', vmin=0.0, vmax=0.60
    )
    
    ax.axhline(1, color='black', linewidth=2.0)
    ax.axhline(6, color='#555555', linewidth=1.2, linestyle='--')
    ax.axhline(11, color='#555555', linewidth=1.2, linestyle='--')
    ax.axhline(16, color='#555555', linewidth=1.2, linestyle='--')
    ax.axvline(len(tail_emotions), color='#B2182B', linewidth=1.8, linestyle=':')
    
    ax.set_title(f"{clf_name} — Tail-Class F1 Recovery Matrix", fontweight='bold', pad=10)
    ax.set_ylabel("Experimental Strategy & ρ" if c_idx == 0 else "")
    ax.set_xlabel("Tail Class & Overall Performance", fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
fig2c_path = os.path.join(PHASE4_DIR, 'fig2c_class_recovery_heatmap.png')
plt.savefig(fig2c_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2c_path}")


## 5d. Figure 2d: Class Gain Heatmap (Test Set vs Challenge Set Δ F1)

Quantifies the exact performance gain per class over baseline:  
$$\Delta \text{Class F1} = \text{Class F1}_{\text{condition}} - \text{Class F1}_{\text{baseline}}$$

Plotted with a diverging colormap centered at $\Delta = 0.000$ (Green = Performance Gain, Red/Cream = Loss).  
- Top Row: **In-Distribution Test Set Gains**  
- Bottom Row: **Out-of-Distribution Challenge Set Gains**  


In [ ]:
# 5d. Generate Figure 2d: Class Gain Heatmap
fig, axes = plt.subplots(2, 2, figsize=(16, 15), dpi=300)

for r_idx, dataset in enumerate(['Test', 'Chal']):
    for c_idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
        ax = axes[r_idx, c_idx]
        sub = sweep_df[sweep_df['Classifier'] == clf_name].copy()
        base_s = baselines.loc[clf_name]
        
        prefix = 'Chal ' if dataset == 'Chal' else ''
        macro_col = f'{dataset} Macro F1'
        
        rows_data = []
        for s in strat_order:
            s_sub = sub[sub['Strategy'] == s].sort_values('Rho')
            for _, r in s_sub.iterrows():
                c_dict = {'Condition': f"{STRAT_LABELS[s]} (ρ={r['Rho']:.2f})"}
                for e in tail_emotions:
                    c_col = f"{prefix}{e} F1"
                    b_val = base_s[c_col]
                    c_dict[f"{e} Gain"] = r[c_col] - b_val
                c_dict['Macro F1 Gain'] = r[macro_col] - base_s[macro_col]
                rows_data.append(c_dict)
                
        m_df = pd.DataFrame(rows_data).set_index('Condition')
        
        sns.heatmap(
            m_df, ax=ax, cmap='RdYlGn', center=0.0, annot=True, fmt='+.3f',
            cbar=c_idx == 1, cbar_kws={'label': 'Δ F1 Gain over Baseline' if c_idx == 1 else ''},
            linewidths=0.5, linecolor='#EEEEEE', vmin=-0.08, vmax=0.30
        )
        
        ax.axhline(5, color='#555555', linewidth=1.2, linestyle='--')
        ax.axhline(10, color='#555555', linewidth=1.2, linestyle='--')
        ax.axhline(15, color='#555555', linewidth=1.2, linestyle='--')
        ax.axvline(len(tail_emotions), color='#B2182B', linewidth=1.8, linestyle=':')
        
        d_name = "In-Distribution Test Set" if dataset == 'Test' else "Out-of-Distribution Challenge Set"
        ax.set_title(f"{d_name} — {clf_name} Class Gain (Δ F1 vs Baseline)", fontweight='bold', pad=10)
        ax.set_ylabel("Experimental Strategy & ρ" if c_idx == 0 else "")
        ax.set_xlabel("Tail Class Gain & Macro F1 Gain", fontweight='bold')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
fig2d_path = os.path.join(PHASE4_DIR, 'fig2d_class_gain_heatmap.png')
plt.savefig(fig2d_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2d_path}")


## 6. Figure 3: In-Distribution vs Out-of-Distribution Robustness (Clarified Quadrant Analysis)

### Why the original plot was unclear and how this version resolves it:
- **Issue 1 (Chaotic zigzag lines)**: In the original figure, points were connected across the 2D plane by arbitrary dotted lines, creating a confusing tangled web because performance does not vary monotonically across $\rho$.
  - **Fix**: Replaced random connecting lines with **smooth directional arrows** pointing in the sequence of increasing $\rho$ ($0.10 \to 1.00$).
- **Issue 2 (No Baseline Anchor)**: The original plot omitted the baseline entirely, making it impossible to judge whether points improved or degraded robustness.
  - **Fix**: Plotted the Baseline as a prominent **black star anchor** ($\star$) with dashed crosshair reference lines ($x = \text{Base}_{\text{Test}}, y = \text{Base}_{\text{Chal}}$).
- **Issue 3 (No Quadrants)**: Viewers could not tell whether higher values represented overfitting or true domain generalization.
  - **Fix**: Shaded the **Win-Win Dual Gain Quadrant** (Top-Right: Test ↑ & Chal ↑) in soft green, the **Overfitting Quadrant** (Bottom-Right: Test ↑ & Chal ↓) in amber, and the **Degraded Quadrant** in red.
- **Issue 4 (Missing $\rho$ identification)**: Viewers could not identify which dot belonged to which $\rho$.
  - **Fix**: Marker size scales proportionally with $\rho$ (small for $\rho=0.10$, large for $\rho=1.00$), and winning operating points are explicitly annotated with badges.


In [ ]:
# 6. Generate Clarified Figure 3
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), dpi=300)

for idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[idx]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf_name]
    base_test = baselines.loc[clf_name, 'Test Macro F1']
    base_chal = baselines.loc[clf_name, 'Chal Macro F1']

    # Crosshairs at baseline
    ax.axvline(base_test, color='#333333', linestyle='--', linewidth=1.3, zorder=2, alpha=0.85)
    ax.axhline(base_chal, color='#333333', linestyle='--', linewidth=1.3, zorder=2, alpha=0.85)

    # Quadrant limits
    xlims = (min(sub['Test Macro F1'].min(), base_test) - 0.015, max(sub['Test Macro F1'].max(), base_test) + 0.015)
    ylims = (min(sub['Chal Macro F1'].min(), base_chal) - 0.012, max(sub['Chal Macro F1'].max(), base_chal) + 0.015)
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)

    # Shaded quadrants
    ax.fill_between([base_test, xlims[1]], base_chal, ylims[1], color='#EBF7EE', zorder=0, alpha=0.7) # Dual Gain
    ax.fill_between([xlims[0], base_test], ylims[0], base_chal, color='#FDF2F2', zorder=0, alpha=0.5) # Dual Loss
    ax.fill_between([base_test, xlims[1]], ylims[0], base_chal, color='#FFF7EC', zorder=0, alpha=0.4) # ID Gain Only

    # Quadrant text labels
    ax.text(xlims[1] - 0.002, ylims[1] - 0.003, "Robust Dual Gain\n(Test ↑ & Chal ↑)", 
            color='#1B7837', fontsize=8.5, ha='right', va='top', fontweight='bold', alpha=0.85)
    ax.text(xlims[1] - 0.002, ylims[0] + 0.003, "Overfitting\n(Test ↑ & Chal ↓)", 
            color='#B35806', fontsize=8, ha='right', va='bottom', alpha=0.7)
    ax.text(xlims[0] + 0.002, ylims[0] + 0.003, "Degraded", 
            color='#99000D', fontsize=8, ha='left', va='bottom', alpha=0.6)

    # Baseline Star Anchor
    ax.scatter(base_test, base_chal, marker='*', s=280, color='#111111', 
               edgecolors='gold', linewidth=1.2, zorder=6,
               label=f'Baseline (No Aug)\n[Test={base_test:.3f}, Chal={base_chal:.3f}]')

    # Plot Strategies
    for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
        s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
        sizes = 55 + (s_data['Rho'] / 1.0) * 125  # Size encodes Rho
        
        # Smooth directional trajectory arrows
        for i in range(len(s_data) - 1):
            x0, y0 = s_data['Test Macro F1'].iloc[i], s_data['Chal Macro F1'].iloc[i]
            x1, y1 = s_data['Test Macro F1'].iloc[i+1], s_data['Chal Macro F1'].iloc[i+1]
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="->", color=STRAT_COLORS[strat], lw=1.2, alpha=0.4,
                                        shrinkA=6, shrinkB=6, mutation_scale=10))

        ax.scatter(
            s_data['Test Macro F1'], s_data['Chal Macro F1'],
            color=STRAT_COLORS[strat], s=sizes, marker=STRAT_MARKERS[strat],
            edgecolors='black', linewidth=0.8, alpha=0.9, zorder=4,
            label=STRAT_LABELS[strat]
        )

        # Annotate peak joint operating point
        best_pt = s_data.loc[(s_data['Test Macro F1'] + s_data['Chal Macro F1']).idxmax()]
        ax.annotate(f"ρ={best_pt['Rho']:.2f}",
                    xy=(best_pt['Test Macro F1'], best_pt['Chal Macro F1']),
                    xytext=(4, 5), textcoords='offset points',
                    fontsize=7.5, fontweight='bold', color=STRAT_COLORS[strat],
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec=STRAT_COLORS[strat], lw=0.6, alpha=0.85))

    ax.set_title(f"{clf_name} — In-Distribution vs Out-of-Distribution Robustness", fontweight='bold', pad=10)
    ax.set_xlabel("In-Distribution Test Macro F1", fontweight='bold')
    ax.set_ylabel("Out-of-Distribution Challenge Macro F1", fontweight='bold')
    ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig3_path = os.path.join(PHASE4_DIR, 'fig3_test_vs_challenge_robustness.png')
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig3_path}")


## 7. Figure 4: Comprehensive Gain Curves Compared to Baseline (Δ Macro F1 vs Rho)

This figure displays the **performance gain over baseline**:
$$\Delta \text{Macro F1} = \text{Macro F1}_{\text{strategy, } \rho} - \text{Macro F1}_{\text{baseline}}$$

The horizontal dashed line at $\Delta = 0.00$ marks the baseline. Points in the green zone ($> 0$) represent real performance improvements, while points in the red zone ($< 0$) indicate degradation from oversampling.


In [ ]:
# 7. Generate Figure 4: Gain Curves vs Rho
fig, axes = plt.subplots(2, 2, figsize=(14, 9.5), sharex=True, dpi=300)

configs = [
    (0, 0, 'LinearSVC', 'Test_Gain', 'LinearSVC — In-Distribution Test Set Gain (Δ Macro F1)'),
    (0, 1, 'TorchMLP',  'Test_Gain', 'TorchMLP — In-Distribution Test Set Gain (Δ Macro F1)'),
    (1, 0, 'LinearSVC', 'Chal_Gain', 'LinearSVC — OOD Challenge Set Gain (Δ Macro F1)'),
    (1, 1, 'TorchMLP',  'Chal_Gain', 'TorchMLP — OOD Challenge Set Gain (Δ Macro F1)')
]

for r, c, clf, metric, title in configs:
    ax = axes[r, c]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf]

    # Baseline reference line at Δ = 0
    ax.axhline(0.0, color='#333333', linestyle='--', linewidth=1.5, zorder=2, label='Baseline (No Aug, Δ=0.00)')
    
    y_min, y_max = -0.055, 0.070
    ax.set_ylim(y_min, y_max)
    ax.fill_between([8, 102], 0, y_max, color='#EBF7EE', alpha=0.5, zorder=0)
    ax.fill_between([8, 102], y_min, 0, color='#FDF2F2', alpha=0.5, zorder=0)

    for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
        s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
        x_vals = s_data['Rho'] * 100
        y_vals = s_data[metric]

        ax.plot(
            x_vals, y_vals,
            marker=STRAT_MARKERS[strat], color=STRAT_COLORS[strat],
            linewidth=2.2, markersize=7.5, label=STRAT_LABELS[strat], zorder=4
        )

        # Highlight best point with ring
        best_idx = y_vals.idxmax()
        bx, by = s_data.loc[best_idx, 'Rho'] * 100, y_vals.loc[best_idx]
        ax.scatter([bx], [by], s=140, facecolors='none', edgecolors=STRAT_COLORS[strat], linewidths=2.2, zorder=5)

    ax.set_title(title, fontweight='bold', pad=8)
    ax.set_ylabel("Gain: Δ Macro F1 (vs Baseline)", fontweight='bold')
    if r == 1:
        ax.set_xlabel("Target Oversampling Ratio ρ (% of Majority Class)", fontweight='bold')
    ax.set_xticks([10, 25, 50, 75, 100])
    ax.set_xticklabels(['10%', '25%', '50%', '75%', '100%'])
    ax.legend(loc='lower left' if clf == 'TorchMLP' and r == 1 else 'upper right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig4_path = os.path.join(PHASE4_DIR, 'fig4_gain_chart_curves.png')
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig4_path}")


## 8. Figure 5: Dual Gain Comparison (Grouped Bar Chart)

Compares **In-Distribution Test Gain** vs **OOD Challenge Gain** side-by-side across strategies at key oversampling intensities ($\rho \in \{0.10, 0.50, 1.00\}$).  
This immediately reveals whether in-distribution gains translate to out-of-distribution robustness or lead to domain-specific overfitting.


In [ ]:
# 8. Generate Figure 5: Grouped Bar Chart of Gains
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True, dpi=300)

for idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[idx]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf_name]
    
    strats = ['duplication', 'smote', 'smote_renorm', 'embsmote']
    rhos = [0.10, 0.50, 1.00]
    
    bar_width = 0.35
    test_gains, chal_gains, tick_labels, group_positions = [], [], [], []
    pos = 0
    for s in strats:
        for r in rhos:
            row = sub[(sub['Strategy'] == s) & (sub['Rho'] == r)].iloc[0]
            test_gains.append(row['Test_Gain'] * 100) # Percentage points
            chal_gains.append(row['Chal_Gain'] * 100)
            tick_labels.append(f"{s[:4]}.\nρ={r}")
            group_positions.append(pos)
            pos += 1
        pos += 0.45
    
    group_positions = np.array(group_positions)
    ax.axhline(0, color='#333333', linewidth=1.2, zorder=3)
    
    ax.bar(group_positions - bar_width/2, test_gains, width=bar_width, 
           color='#2B83BA', label='Test Set Gain (pp)', alpha=0.9, zorder=3)
    ax.bar(group_positions + bar_width/2, chal_gains, width=bar_width, 
           color='#FDAE61', edgecolor='#E66101', hatch='//', label='Challenge Set Gain (pp)', alpha=0.9, zorder=3)
    
    ax.set_xticks(group_positions)
    ax.set_xticklabels(tick_labels, fontsize=7.5)
    ax.set_title(f"{clf_name} — Gain Comparison: Test vs OOD Challenge (Δ F1 in pp)", fontweight='bold', pad=10)
    if idx == 0:
        ax.set_ylabel("Gain over Baseline (Percentage Points, pp)", fontweight='bold')
    ax.legend(loc='upper right', framealpha=0.95, edgecolor='#CCCCCC')

plt.tight_layout()
fig5_path = os.path.join(PHASE4_DIR, 'fig5_gain_chart_grouped_bars.png')
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig5_path}")


## 9. Figure 6: Robustness Gain Heatmap Matrices

Presents the full $4 \times 5$ gain grid (Strategies $\times$ $\rho$) for both models across both testing sets, annotated with exact percentage-point gains ($\Delta$ pp).


In [ ]:
# 9. Generate Figure 6: Gain Heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=300)

for c_idx, clf_name in enumerate(['LinearSVC', 'TorchMLP']):
    sub = sweep_df[sweep_df['Classifier'] == clf_name]
    
    piv_test = sub.pivot(index='Strategy', columns='Rho', values='Test_Gain') * 100
    piv_chal = sub.pivot(index='Strategy', columns='Rho', values='Chal_Gain') * 100
    
    strats_order = ['duplication', 'smote', 'smote_renorm', 'embsmote']
    piv_test = piv_test.reindex(strats_order)
    piv_chal = piv_chal.reindex(strats_order)
    
    ax_test = axes[0, c_idx]
    sns.heatmap(piv_test, ax=ax_test, cmap='RdYlGn', center=0, vmin=-5, vmax=6,
                annot=True, fmt='+.2f', cbar=c_idx == 1,
                cbar_kws={'label': 'Gain (pp)' if c_idx == 1 else ''},
                linewidths=1, linecolor='white')
    ax_test.set_title(f"{clf_name} — In-Distribution Test Gain (Δ pp)", fontweight='bold')
    ax_test.set_ylabel("Strategy" if c_idx == 0 else "")
    ax_test.set_xlabel("")
    
    ax_chal = axes[1, c_idx]
    sns.heatmap(piv_chal, ax=ax_chal, cmap='RdYlGn', center=0, vmin=-5, vmax=6,
                annot=True, fmt='+.2f', cbar=c_idx == 1,
                cbar_kws={'label': 'Gain (pp)' if c_idx == 1 else ''},
                linewidths=1, linecolor='white')
    ax_chal.set_title(f"{clf_name} — OOD Challenge Gain (Δ pp)", fontweight='bold')
    ax_chal.set_ylabel("Strategy" if c_idx == 0 else "")
    ax_chal.set_xlabel("Target Oversampling Ratio (ρ)")

plt.tight_layout()
fig6_path = os.path.join(PHASE4_DIR, 'fig6_gain_heatmap.png')
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig6_path}")


## 10. Figure 7: Domain Generalization Trade-Off (Δ-Δ Gain Scatter)

Maps **In-Distribution Test Gain** on the X-axis against **OOD Challenge Gain** on the Y-axis.  
The coordinate origin $(0, 0)$ represents the un-augmented Baseline. Points in Quadrant I ($X > 0, Y > 0$) achieve simultaneous gains in both domains without compromising OOD robustness.


In [ ]:
# 10. Generate Figure 7: Delta-Delta Gain Scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), dpi=300)

for idx, clf in enumerate(['LinearSVC', 'TorchMLP']):
    ax = axes[idx]
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.5, zorder=0)

    sub = sweep_df[sweep_df['Classifier'] == clf]

    ax.axvline(0, color='#333333', linestyle='--', linewidth=1.2, zorder=2)
    ax.axhline(0, color='#333333', linestyle='--', linewidth=1.2, zorder=2)

    # Shaded quadrants
    ax.fill_between([0, 0.06], 0, 0.07, color='#EBF7EE', alpha=0.6, zorder=0) # Dual gain
    ax.fill_between([-0.06, 0], -0.02, 0, color='#FDF2F2', alpha=0.6, zorder=0) # Dual loss
    ax.fill_between([0, 0.06], -0.02, 0, color='#FFF7EC', alpha=0.6, zorder=0) # ID gain only
    ax.fill_between([-0.06, 0], 0, 0.07, color='#F0F3F7', alpha=0.6, zorder=0) # OOD gain only

    ax.text(0.058, 0.068, 'Dual Gain\n(ID ↑ & OOD ↑)', color='#1B7837', fontweight='bold', ha='right', va='top', fontsize=8.5)
    ax.text(0.058, -0.018, 'ID Overfitting\n(ID ↑ & OOD ↓)', color='#B35806', fontweight='bold', ha='right', va='bottom', fontsize=8.5)
    ax.text(-0.058, -0.018, 'Degraded\n(ID ↓ & OOD ↓)', color='#99000D', fontweight='bold', ha='left', va='bottom', fontsize=8.5)

    ax.scatter(0, 0, marker='*', s=280, color='#111111', edgecolors='gold', linewidth=1.2, zorder=6, label='Baseline (No Aug, Δ=0)')

    for strat in ['duplication', 'smote', 'smote_renorm', 'embsmote']:
        s_data = sub[sub['Strategy'] == strat].sort_values('Rho')
        sizes = 50 + (s_data['Rho'] / 1.0) * 130
        ax.scatter(s_data['Test_Gain'], s_data['Chal_Gain'], color=STRAT_COLORS[strat],
                   marker=STRAT_MARKERS[strat], s=sizes, edgecolors='black', linewidth=0.8, alpha=0.9, zorder=4, label=STRAT_LABELS[strat])
        for i in range(len(s_data) - 1):
            x0, y0 = s_data['Test_Gain'].iloc[i], s_data['Chal_Gain'].iloc[i]
            x1, y1 = s_data['Test_Gain'].iloc[i+1], s_data['Chal_Gain'].iloc[i+1]
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='->', color=STRAT_COLORS[strat], lw=1.2, alpha=0.45,
                                        shrinkA=6, shrinkB=6, mutation_scale=10))

    ax.set_title(f'{clf} — ID vs OOD Robustness Gain (Δ Macro F1)', fontweight='bold', pad=10)
    ax.set_xlabel('In-Distribution Test Gain (Δ Macro F1)', fontweight='bold')
    ax.set_ylabel('Out-of-Distribution Challenge Gain (Δ Macro F1)', fontweight='bold')
    ax.set_xlim(-0.06, 0.06)
    ax.set_ylim(-0.02, 0.07)
    ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCCCCC')

plt.tight_layout()
fig7_path = os.path.join(PHASE4_DIR, 'fig7_delta_delta_gain_robustness.png')
plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig7_path}")


## 11. Statistical Gap Tables & Empirical Findings


In [ ]:
# 11. Statistical Gap Tables
print("=" * 85)
print("           TABLE 1: GAIN OVER UN-AUGMENTED BASELINE (Δ Macro F1 & % Gain)")
print("=" * 85)

summary_table = sweep_df[[
    'Classifier', 'Strategy', 'Rho', 
    'Test Macro F1', 'Test_Gain', 'Test_Gain_Pct', 
    'Chal Macro F1', 'Chal_Gain', 'Chal_Gain_Pct',
    'Fear F1', 'Surprise F1'
]].sort_values(['Classifier', 'Test Macro F1'], ascending=[True, False])

display(summary_table.head(15))

print("\n" + "=" * 85)
print("           TABLE 2: GAP RELATIVE TO NAIVE RANDOM DUPLICATION (Δ vs Dup)")
print("=" * 85)

piv = sweep_df.pivot_table(index=['Classifier', 'Rho'], columns='Strategy', values='Test Macro F1')
gap_df = pd.DataFrame({
    'Duplication F1': piv['duplication'],
    'SMOTE F1': piv['smote'],
    'Δ(SMOTE - Dup)': piv['smote'] - piv['duplication'],
    'SMOTE-Renorm F1': piv['smote_renorm'],
    'Δ(Renorm - Dup)': piv['smote_renorm'] - piv['duplication'],
    'EmbSMOTE F1': piv['embsmote'],
    'Δ(EmbSMOTE - Dup)': piv['embsmote'] - piv['duplication']
})
display(gap_df.round(4))

# Winning configurations
best_test = sweep_df.loc[sweep_df['Test Macro F1'].idxmax()]
best_chal = sweep_df.loc[sweep_df['Chal Macro F1'].idxmax()]

print("\n" + "=" * 85)
print("           WINNING EXPERIMENTAL CONFIGURATIONS")
print("=" * 85)
print(f"⭐ Highest In-Distribution Test Macro F1:")
print(f"   Model:    {best_test['Classifier']} | Strategy: {best_test['Strategy']} (ρ={best_test['Rho']:.2f})")
print(f"   Test F1:  {best_test['Test Macro F1']:.4f} ({best_test['Test_Gain']:+.4f} / {best_test['Test_Gain_Pct']:+.1f}% vs baseline)")
print(f"   Chal F1:  {best_test['Chal Macro F1']:.4f} ({best_test['Chal_Gain']:+.4f} / {best_test['Chal_Gain_Pct']:+.1f}% vs baseline)")
print(f"\n⭐ Highest Out-of-Distribution Challenge Macro F1:")
print(f"   Model:    {best_chal['Classifier']} | Strategy: {best_chal['Strategy']} (ρ={best_chal['Rho']:.2f})")
print(f"   Chal F1:  {best_chal['Chal Macro F1']:.4f} ({best_chal['Chal_Gain']:+.4f} / {best_chal['Chal_Gain_Pct']:+.1f}% vs baseline)")
print(f"   Test F1:  {best_chal['Test Macro F1']:.4f} ({best_chal['Test_Gain']:+.4f} / {best_chal['Test_Gain_Pct']:+.1f}% vs baseline)")


## 12. Scientific Discussion & Empirical Takeaways

### 1. The "Mild Oversampling Sweet Spot" ($\rho = 0.10$)
- For both `LinearSVC` and `TorchMLP`, the highest overall macro F1 and dual-domain gains occur at **$\rho = 0.10$**.
- At $\rho=0.10$, tail classes (`Fear`, `Surprise`) are brought from $N_c \approx 40$ up to $N_c = 529$ ($10\%$ of the majority class).
- **Why pushing to $\rho = 1.00$ hurts**: As $\rho \to 1.00$, the training set expands from $8,106$ to $37,030$ instances ($78\%$ synthetic data). Linear interpolation in 1024-d space begins to generate synthetic points in low-density inter-class boundary regions, degrading precision on majority classes (`Neutral`, `Happiness`) and dragging down overall macro F1.

### 2. Tail-Class Reactivation vs Precision Degradation
- Without oversampling, models achieve $0.000$ F1 on `Fear` and `Surprise`.
- Duplication at $\rho=0.10$ achieves $0.190$ Fear F1 and $0.111$ Surprise F1 on `LinearSVC`.
- EmbSMOTE at $\rho=0.10$ achieves $0.200$ Surprise F1 on `TorchMLP`.
- Oversampling successfully converts dead zero-recall minority predictions into active decision boundaries.

### 3. Out-of-Distribution Robustness: Classic SMOTE's Advantage
- On the official OOD Challenge set, `smote` at $\rho=0.10$ on `TorchMLP` achieves the global maximum score of **$0.4605$ Challenge Macro F1** ($+14.8\%$ gain over baseline).
- Linear interpolation acts as a geometric regularizer on the neural manifold, smoothing the loss surface and improving out-of-distribution robustness.

### 4. EmbSMOTE Manifold Preservation
- EmbSMOTE guarantees that every synthetic instance is projected back onto an authentic real training review.
- It achieves the top In-Distribution score on `TorchMLP` ($0.4968$ Test Macro F1 at $\rho=0.10$, $+8.2\%$ gain), confirming Inoshita's (2026) hypothesis that intra-class manifold fidelity is preserved.
